# Pipeline 04Gf: generation variance (P1-2)

The G2a headline result is a **null finding**: no pairwise modality difference in judge
faithfulness survives Holm correction (all `p_adj = 1.0`, Cliff's *d* negligible). That
finding was produced with `n_generations = 1` at `temperature = 1.0`, which leaves the
obvious question open — *would a second draw have looked different?* Without a spread
estimate, "no detectable difference" cannot be separated from "one noisy draw each".

**This notebook is self-contained: generation *and* both judges run here.** There is no
follow-up evaluation notebook — `05G` scores `results/global/` and is untouched by this
run, which writes only to `results/global_variance*`.

How it is set up:

| Choice | Why |
| --- | --- |
| **Subset, not all 72** | The three features with rubric headroom (`windspeed`, `holiday`, `weekday` - the near-flat stratum, mean totals 0.722/0.722/0.667) plus `hr` as a **ceiling control** x 2 XAI models x 3 LLM forms = **24 cells** |
| **The existing G2a record is generation 0** | It was produced by the identical prompt and code path, so it is a valid draw. Only 2 further draws per cell are billed -> **48 calls**, not 72 |
| **Scored by the rubric *and* the judge** | The rubric is deterministic, so its spread is pure generation variance — but it is coarse (four values; one sub-score flip moves the total by 3x the effect to be resolved). The judge resolves finer. Measured against the actual token costs, the judge is not the expensive part: generation is **~$10** (json 82k + tooluse 69k input per call), both vendors of judge together **~$4** |
| **Writes to `results/global_variance/`** | The approved G2a artefacts under `results/global/` stay byte-identical and frozen |

The template form is excluded on purpose: it is deterministic, so its variance is zero by
construction and a run would only cost tokens.

**Cost, derived from this project's own billed runs** (`results/eval_summary.csv` carries
`cost_usd` against known token counts, which pins the rate empirically rather than from a
price list): 48 generation calls ~ **$10.3** (json $5.4, tool-use $4.6, vision $0.2 — the
input payloads dominate) plus 96 judge calls ~ **$4**. About **$14-16** in total.

**Why the subset is chosen by headroom, not by shape type.** The rubric total takes only
four distinct values across the 72 G2a records (0.500 / 0.667 / 0.833 / 1.000), and **48 of
those 72 sit at 1.000**. Five features (`hr`, `hum`, `mnth`, `temp`, `weathersit`) score a
flat 1.000 in every LLM cell — their variance is zero by construction, and drawing them
again would buy a foregone conclusion. Every between-modality difference reported in `05G`
originates in the near-flat stratum, so that is the only place where a spread estimate can
be informative. `hr` rides along as a ceiling control.

**What to report from this.** The within-cell spread of the rubric total against the
between-modality differences observed in `05G` (0.889 / 0.898 / 0.898 / 0.944 — a range of
about **0.055**). If the spread is of the same order, the null finding is "underpowered
against sampling noise" and must be stated as such; if it is clearly smaller, the null
finding stands on firmer ground.

**Read the coarseness as a limitation of this measurement, not as a result.** One rubric
sub-score flipping from 1.0 to 0.5 moves the total by 0.167 — three times the entire
between-modality range. So the rubric can only report "no draw changed a sub-score" or
"the spread dwarfs the effect"; it cannot resolve anything in between. A spread of zero
here means *the rubric-visible content was stable*, not *the texts were identical*.

> **`RUN_API` guard.** Default `False`: the notebook verifies the whole non-API path
> (subset selection, generation-0 reuse, rubric scoring, the spread table) against the
> records already on disk and writes **nothing**. Set `True` for the billed run.


In [ ]:
from __future__ import annotations

import sys, json, time, shutil
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import (
    EXPLANATIONS_DIR, RESULTS_DIR, PROMPTS_DIR, GLOBAL_RESULTS_SUBDIR,
    list_global_features, feature_importance_map,
    build_feature_json_payload, assemble_global_system_prompt, shape_plot_path,
    build_global_record, run_resumable_global_generation, seed_generation_zero,
    GlobalToolBox, run_global_tool_use_loop,
)
from utils.llm import (
    ask_text, ask_with_images, _get_client, DEFAULT_MODEL,
    MAX_TOKENS_GENERATION, strip_scratchpad,
)
from utils import rubric

# --- config -----------------------------------------------------------------
RUN_API    = False            # <- True to draw the extra generations (48 calls)
RUN_JUDGE  = False            # <- True to score them with both vendors (96 calls)
LOSS_KEY   = 'poisson_log'
MODEL      = DEFAULT_MODEL
MAX_TOKENS = MAX_TOKENS_GENERATION
XAI_MODELS = ['xgb', 'ebm']

# Chosen by rubric HEADROOM, not by shape-type coverage. Five of the nine features
# (hr, hum, mnth, temp, weathersit) score a flat 1.000 in every LLM cell: they are at
# the rubric ceiling, so their variance is ~0 by construction and measuring it costs
# tokens for a foregone conclusion. All the between-modality variation in 05G comes
# from the near-flat stratum, so that is where the spread has to be measured.
#
#   windspeed  near-flat, mean 0.722, 0 of 6 cells at ceiling  <- most headroom
#   holiday    near-flat, mean 0.722, 1 of 6 at ceiling
#   weekday    near-flat (ebm) / categorical (xgb), mean 0.667, 1 of 6 at ceiling
#   hr         categorical, 6 of 6 at ceiling                  <- deliberate control
#
# hr is kept as a CONTROL: if even a perfect-scoring cell moves between draws, the
# instrument is noisier than assumed; if it stays at 1.000, the spread measured in the
# other three is attributable to the hard cells rather than to scorer flakiness.
SUBSET_FEATURES = ['windspeed', 'holiday', 'weekday', 'hr']
FORMS           = ['json', 'vision', 'tooluse']   # template is deterministic -> var = 0
K               = 3                               # draws per cell, incl. generation 0

PLOTS_DIR   = EXPLANATIONS_DIR / 'plots' / 'global'
FROZEN_DIR  = RESULTS_DIR / GLOBAL_RESULTS_SUBDIR      # the approved G2a records
OUT_DIR     = RESULTS_DIR / 'global_variance'          # this notebook's own directory
FROZEN_JUDGE = {'anthropic': RESULTS_DIR / 'global_judge',
                'openai':    RESULTS_DIR / 'global_judge_openai'}
JUDGE_DIR    = {'anthropic': RESULTS_DIR / 'global_variance_judge',
                'openai':    RESULTS_DIR / 'global_variance_judge_openai'}

ALL_FEATURES = list_global_features('ebm', explanations_dir=EXPLANATIONS_DIR)
assert set(SUBSET_FEATURES) <= set(ALL_FEATURES), 'unknown feature in SUBSET_FEATURES'

n_cells = len(SUBSET_FEATURES) * len(XAI_MODELS) * len(FORMS)
print(f'LLM model: {MODEL}')
print(f'Subset:    {SUBSET_FEATURES} x {XAI_MODELS} x {FORMS} = {n_cells} cells')
print(f'Draws:     k={K} per cell (generation 0 reused from {FROZEN_DIR.name}/)')
print(f'Billed:    {n_cells * (K - 1)} generation calls'
      + (f' + {n_cells * (K - 1) * 2} judge calls' if RUN_JUDGE else ''))
print(f'Output:    {OUT_DIR}')
print(f'RUN_API:   {RUN_API}  ->  ' + ('BILLED real run' if RUN_API
      else 'verification only, nothing written'))
print(f'RUN_JUDGE: {RUN_JUDGE}')

## 1. Seed the runs with generation 0 from the frozen G2a records

Each `results/global/{form}_{model}_{feature}.json` was produced by the same prompt and the
same code path as the draws below, so it is a legitimate first draw. Copying it in as
`_gen0` means only `k-1` draws per cell are billed — and it anchors the spread to the exact
records the reported G2a numbers came from.


In [ ]:
# seed_generation_zero (utils.global_feature) is idempotent: an existing gen0 file is
# never replaced, so a re-run cannot swap out a draw the later draws were compared against.
missing = [f'{form}_{m}_{x}.json'
           for form in FORMS for m in XAI_MODELS for x in SUBSET_FEATURES
           if not (FROZEN_DIR / f'{form}_{m}_{x}.json').exists()]
print(f'all {n_cells} frozen source records present: {not missing}')
if missing:
    print('  missing (run 04Gb/04Gc/04Gd first):', missing)

if RUN_API and not missing:
    seeded = []
    for form in FORMS:
        seeded += seed_generation_zero(
            form=form, model_names=XAI_MODELS, features=SUBSET_FEATURES,
            src_dir=FROZEN_DIR, out_dir=OUT_DIR, n_generations=K)
    print(f'Seeded {len(seeded)} generation-0 record(s) into {OUT_DIR}')

    # Generation 0 is byte-identical to the frozen record, and the judge runs at
    # temperature 0.0 - so its verdict is already paid for. Copy those in as well
    # rather than re-buying 48 judgements of text that has not changed.
    for vendor, src in FROZEN_JUDGE.items():
        got = []
        for form in FORMS:
            got += seed_generation_zero(
                form=form, model_names=XAI_MODELS, features=SUBSET_FEATURES,
                src_dir=src, out_dir=JUDGE_DIR[vendor], n_generations=K)
        print(f'  seeded {len(got)} generation-0 judge verdict(s) ({vendor})')
elif not RUN_API:
    print('RUN_API=False -> nothing copied.')


## 2. Draw the remaining generations

One dispatcher per form, identical to `04Gb`/`04Gc`/`04Gd` — same prompts, same payloads,
same parameters. The only difference is that `run_resumable_global_generation` is called
with `n_generations=K`, so the filenames carry a `_gen{idx}` suffix and generation 0 (the
seeded record) is skipped by the resume check rather than regenerated.


In [ ]:
SYSTEM = {(form, m): assemble_global_system_prompt(form, m, prompts_dir=PROMPTS_DIR)
          for form in FORMS for m in XAI_MODELS}
RANKS  = {m: feature_importance_map(m, explanations_dir=EXPLANATIONS_DIR) for m in XAI_MODELS}
client = _get_client() if RUN_API else None

USER_TMPL = 'Describe the global effect of the feature "{feature}" on hourly bike demand.'


def _generate(form, model_name, feature, gen_idx):
    system, t0 = SYSTEM[(form, model_name)], time.time()
    extra, stop = None, None
    try:
        if form == 'json':
            payload = build_feature_json_payload(model_name, feature,
                                                 explanations_dir=EXPLANATIONS_DIR)
            resp = ask_text(USER_TMPL.format(feature=feature) + '\n\n'
                            + json.dumps(payload, indent=2),
                            system=system, model=MODEL, max_tokens=MAX_TOKENS,
                            cache_system=True)
            text, usage = strip_scratchpad(resp['content'][0]['text']), resp.get('usage', {})
            stop = resp.get('stop_reason')
        elif form == 'vision':
            plot = shape_plot_path(model_name, feature, plots_dir=PLOTS_DIR)
            rank = RANKS[model_name][feature]['rank']
            user = (USER_TMPL.format(feature=feature)
                    + f'\nIts global importance rank is {rank} of {len(ALL_FEATURES)}.'
                    + "\nThe attached image is this feature's global plot.")
            resp = ask_with_images(user, [plot], system=system, model=MODEL,
                                   max_tokens=MAX_TOKENS, cache_system=True)
            text, usage = strip_scratchpad(resp['content'][0]['text']), resp.get('usage', {})
            stop = resp.get('stop_reason')
            extra = {'plot_file': plot.name}
        else:
            toolbox = GlobalToolBox(model_name, explanations_dir=EXPLANATIONS_DIR,
                                    plots_dir=PLOTS_DIR, loss_key=LOSS_KEY)
            user = (USER_TMPL.format(feature=feature) + ' Also state how important it is '
                    'relative to the other features. Retrieve whatever you need with the '
                    'tools before answering.')
            text, call_log, in_tok, out_tok, stop = run_global_tool_use_loop(
                client, toolbox, user_message=user, system=system, model=MODEL,
                max_tokens=MAX_TOKENS)
            text, usage = strip_scratchpad(text), {'input_tokens': in_tok,
                                                   'output_tokens': out_tok}
            extra = {'n_tool_calls': len(call_log), 'tool_calls': call_log}
    except Exception as e:
        print(f'  [ERROR] {form} {model_name} {feature} gen{gen_idx}: '
              f'{type(e).__name__}: {e} -> skip')
        return None

    rec = build_global_record(
        form=form, model_name=model_name, feature=feature, explanation=text,
        usage=usage, llm_model=MODEL, loss_key=LOSS_KEY,
        elapsed_s=round(time.time() - t0, 2),
        include_cache=(form != 'tooluse'), extra=extra)
    rec['generation_idx'] = gen_idx
    rec['stop_reason'] = stop
    rec['max_tokens'] = MAX_TOKENS
    # Same trap as P0-1 in 04Ge: at temperature 1.0 a draw can spend the whole budget
    # inside the scratchpad and emit no [EFFECT]/[IMPORTANCE] block at all. Its rubric
    # score would then be ~0 and would land in the spread as if it were generation
    # variance. Flag it here; section 3 excludes flagged draws from the headline.
    truncated = stop == 'max_tokens' or rec['usage']['output_tokens'] >= MAX_TOKENS
    print(f"  {form:8}{model_name.upper():4} {feature:9} gen{gen_idx} "
          f"out={rec['usage']['output_tokens']}/{MAX_TOKENS} stop={stop}"
          + ('   <-- TRUNCATED, excluded from the spread' if truncated else ''))
    return rec


if RUN_API:
    records = []
    for form in FORMS:
        print(f'--- {form} ---')
        records += run_resumable_global_generation(
            form=form, model_names=XAI_MODELS, features=SUBSET_FEATURES,
            out_dir=OUT_DIR, n_generations=K,
            generate=lambda m, f, g, _form=form: _generate(_form, m, f, g))
    print(f'\n{len(records)} record(s) total ({n_cells} cells x k={K}).')
else:
    print('RUN_API=False -> no calls. Set True to draw the remaining generations.')

## 3. Spread of the rubric total within each cell (the coarse, deterministic half)

The deterministic rubric scores every draw at zero cost. The question this table answers:
**is the within-cell spread small relative to the between-modality differences reported in
`05G`?** Those differences are 0.889 / 0.898 / 0.898 / 0.944 on the rubric total — a range
of about **0.055** between the best and worst form. If the mean within-cell standard
deviation is of that order or larger, the modality comparison is not resolvable at k=1 and
the null finding has to be reported as underpowered.


In [ ]:
import pandas as pd

def truncated_draws(src_dir: Path) -> set[str]:
    """Draws that hit the output ceiling - a token artefact, not generation variance.

    A draw cut off mid-scratchpad emits no [EFFECT]/[IMPORTANCE] block at all, so the
    rubric scores it ~0 and the judge marks it wrong. Leaving it in would report that as
    draw-to-draw variability. Legacy draws carry no stop_reason, so the output-token
    count is the fallback signal (same two-signal logic as utils.global_whole).
    """
    out = set()
    for p in sorted(Path(src_dir).glob('*.json')):
        r = json.loads(p.read_text())
        cap = r.get('max_tokens') or MAX_TOKENS
        if r.get('stop_reason') == 'max_tokens' or r['usage'].get('output_tokens', 0) >= cap:
            out.add(p.stem)
    return out


EXCLUDED = truncated_draws(OUT_DIR) if OUT_DIR.exists() else set()
if EXCLUDED:
    print(f'Excluded {len(EXCLUDED)} truncated draw(s) from the spread: '
          f'{sorted(EXCLUDED)}')
    print('  (re-draw them by deleting the file and re-running section 2)\n')


def spread_table(src_dir: Path) -> pd.DataFrame | None:
    files = [p for p in sorted(Path(src_dir).glob('*.json')) if p.stem not in EXCLUDED]
    if not files:
        return None
    rows = [rubric.score_result_file(p) for p in files]
    df = pd.DataFrame(rows)
    g = df.groupby(['form_pipeline', 'xai_model', 'feature'])['total']
    out = g.agg(n='count', mean='mean', sd='std', lo='min', hi='max').reset_index()
    out['range'] = (out['hi'] - out['lo']).round(3)
    return out.round(3)

spread = spread_table(OUT_DIR)
if spread is None:
    print(f'No records in {OUT_DIR} yet - run section 2 with RUN_API=True.')
else:
    print('Rubric total per cell across draws:')
    print(spread.to_string(index=False))
    print('\nMean within-cell sd by form:')
    print(spread.groupby('form_pipeline')['sd'].mean().round(4))
    # The control cells (hr, at the rubric ceiling in every G2a cell) are reported
    # separately: they say whether the instrument itself moves, which is the baseline
    # the headroom cells have to be read against.
    ctrl = spread[spread.feature == 'hr']
    head = spread[spread.feature != 'hr']
    print(f"\nControl (hr, at the ceiling in G2a): mean sd = {ctrl['sd'].mean():.4f}, "
          f"max range = {ctrl['range'].max():.3f}")
    print(f"Headroom cells (near-flat stratum):  mean sd = {head['sd'].mean():.4f}, "
          f"max range = {head['range'].max():.3f}")

    mean_sd = head['sd'].mean()
    between = 0.944 - 0.889   # best - worst form on the rubric total in 05G
    step    = 1 / 6           # one sub-score flipping 1.0 -> 0.5 moves the total by 0.167
    print(f'\nBetween-modality range in 05G = {between:.4f}')
    print(f'Smallest change this instrument can register = {step:.4f} '
          f'({step / between:.1f}x the effect it would have to resolve)')
    if mean_sd == 0:
        print('Verdict: no draw changed a rubric sub-score. The rubric-visible content is '
              'stable across draws - which does NOT establish that the texts are '
              'equivalent, only that this coarse instrument cannot separate them. Report '
              'as: generation variance is not detectable at rubric granularity.')
    elif mean_sd >= between * 0.5:
        print('Verdict: spread is of the same order as the modality differences -> the '
              'null finding is underpowered against sampling noise; report it as such.')
    else:
        print('Verdict: spread is smaller than the modality differences -> the null '
              'finding is not explained by single-draw sampling noise.')

## 4. Reference-based judge on every draw (both vendors)

The rubric above is deterministic, so its spread is **pure generation variance**. It is
also coarse: the total takes four values, and one sub-score flip moves it by 0.167 — three
times the between-modality range it would have to resolve. The judge is the finer
instrument, at the price of adding a second variance source of its own (it runs at
`temperature = 0.0`, so that contribution is small but not exactly zero).

Reading the two together is what makes this informative:

| rubric spread | judge spread | reading |
| --- | --- | --- |
| 0 | ~0 | draws are genuinely interchangeable; the null finding is not a sampling artefact |
| 0 | large | the texts differ in ways the rubric cannot see — the rubric-only variant would have been **misleading**, and the reported rubric numbers are less stable than they look |
| large | large | the modality comparison is underpowered at k = 1; report the null finding as such |

Generation 0's verdicts were seeded in section 1, so only the new draws are billed:
**24 cells × 2 draws × 2 vendors = 96 judge calls**.


In [ ]:
RUN_JUDGE_HERE = RUN_JUDGE and OUT_DIR.exists() and any(OUT_DIR.glob('*.json'))

if not RUN_JUDGE_HERE:
    print('RUN_JUDGE=False (or no draws yet) -> no judge calls.')
else:
    from utils.eval import run_global_judge
    from utils.llm import ask_text as _ask_anthropic, ask_openai_text, OPENAI_JUDGE_MODEL_FINAL

    ANTHROPIC_JUDGE = 'claude-opus-4-8'
    OPENAI_JUDGE    = OPENAI_JUDGE_MODEL_FINAL

    def _ask_openai(prompt, *, system, model, max_tokens, cache_system=None, temperature=None):
        return ask_openai_text(prompt, system=system, model=model,
                               max_tokens=max_tokens, temperature=temperature)

    # Idempotent: the seeded generation-0 verdicts are loaded, not re-bought.
    judged = {}
    for vendor, ask_fn, jm in (('anthropic', _ask_anthropic, ANTHROPIC_JUDGE),
                               ('openai', _ask_openai, OPENAI_JUDGE)):
        judged[vendor] = run_global_judge(
            ask_fn, jm, src_subdir='global_variance',
            out_subdir=JUDGE_DIR[vendor].name)
        print(f'  {vendor:10} {len(judged[vendor])} verdict(s)')


## 5. Verdict: is the null finding a sampling artefact?

Both spreads side by side, per cell and aggregated. The control feature (`hr`, at the
rubric ceiling in every G2a cell) is reported separately: it is the baseline that says
whether the instruments themselves move.


In [ ]:
def judge_spread(vendor: str) -> pd.DataFrame | None:
    d = JUDGE_DIR[vendor]
    files = sorted(d.glob('*.json')) if d.is_dir() else []
    if not files:
        return None
    rows = []
    for f in files:
        if f.stem in EXCLUDED:          # same truncated draws as in section 3
            continue
        r = json.loads(f.read_text())
        rows.append({'form_pipeline': r['form_pipeline'], 'xai_model': r['xai_model'],
                     'feature': r['feature'], 'faithfulness': r['faithfulness']})
    g = pd.DataFrame(rows).groupby(['form_pipeline', 'xai_model', 'feature'])['faithfulness']
    return g.agg(n='count', mean='mean', sd='std', lo='min', hi='max').reset_index().round(3)


js = {v: judge_spread(v) for v in JUDGE_DIR}
if spread is None or all(x is None for x in js.values()):
    print('Nothing to compare yet - run sections 2 and 4 with the flags on.')
else:
    print('Rubric total (deterministic -> pure generation variance):')
    print(f"  control (hr)        mean sd = {spread[spread.feature == 'hr']['sd'].mean():.4f}")
    print(f"  headroom (near-flat) mean sd = {spread[spread.feature != 'hr']['sd'].mean():.4f}")

    for vendor, t in js.items():
        if t is None:
            continue
        print(f'\nJudge faithfulness, {vendor} (generation + judge variance):')
        print(f"  control (hr)        mean sd = {t[t.feature == 'hr']['sd'].mean():.4f}")
        print(f"  headroom (near-flat) mean sd = {t[t.feature != 'hr']['sd'].mean():.4f}")
        moved = t[t.sd.fillna(0) > 0]
        print(f'  cells that moved between draws: {len(moved)} of {len(t)}')
        if len(moved):
            print(moved.to_string(index=False))

    # The comparison that decides how to report G2a.
    a = js.get('anthropic')
    if a is not None:
        judge_sd = a[a.feature != 'hr']['sd'].mean()
        between  = 4.50 - 4.28      # template - json on judge faithfulness in 05G
        print(f'\nMean within-cell judge sd (headroom) = {judge_sd:.3f}')
        print(f'Between-modality range in 05G        = {between:.3f}')
        if judge_sd >= between:
            print('Verdict: a single draw moves the score by at least as much as the '
                  'entire modality difference. The G2a null finding is UNDERPOWERED '
                  'against sampling noise and must be reported as such.')
        else:
            print('Verdict: draw-to-draw movement is smaller than the modality '
                  'difference. The null finding is not explained by single-draw noise.')
